# PaperRun — consolidated cDTI whole-heart pipeline (inference + evaluation + helix)

**One notebook, one run.** YOLO crop → resize 256 → nnUNet LV (5-fold) → YOLO insertion points →
un-crop → metrics (Dice / Hausdorff / MD / FA / DWI) → **helix angle (CarDpy)** → plots.

### How to use
- **Our 3 runs:** set `DATASET` to `100` / `300` / `310` in the config cell and *Run All*.
  - `100` = SmartHealth (4 contrasts) · `300` = SmartHealth+DirVsAvg (4 contrasts) · `310` = combined **MD+DWI baseline** (2 contrasts, the weights we ship to collaborators).
  - `100` vs `300` = benefit of more data; `300` vs `310` = benefit of more contrasts.
- **Collaborators:** set `COLLAB_MODE=True`, fill in `COLLAB` paths + the baseline weights, and turn off
  stages you can't run (e.g. `RUN_METRICS=False` if you have no ground truth).
- **Stop anywhere:** every stage has a `RUN_*` switch. A collaborator with only the baseline model can run
  `RUN_CROP → RUN_LV → RUN_IP` and stop before metrics/helix.

### Fixes baked in (previously flagged)
1. **Insertion-point overwrite bug** — IP labels used to erase LV (label 1) in the evaluation copy.
   `EVAL_LV_ON_PURE_MASK=True` scores LV Dice/MD/FA/DWI on the **pure LV mask**; IP Hausdorff still comes
   from the combined mask. The notebook prints how many LV pixels an IP *would* have erased.
2. **Dynamic spacing** — the mm/voxel factor is `VOXEL_SPACING_MM` (default 1.25) passed into
   `eval_util.process_folders`, no longer hardcoded inside it. The crop→256 scale factor is already per-case.

> **Helix angle** needs the raw 4D diffusion data (`NifTis` + `.bvals`/`.bvecs`) per volunteer.
> The test-set raw data is staged in `TestRawCases/` (`RAW_DIFFUSION_ROOT`). GT helix uses the native
> full-FoV manual mask (`04_Segmentation_Masks`); predicted helix uses the un-cropped predicted LV.
> **Re-plot / helix only:** turn off GT/CROP/LV/IP/METRICS, set `ALLOW_OVERWRITE=True` — the plot cell reads
> the saved per-run `.xlsx`. Once all three runs exist, it also writes the combined
> 3-run violins to `PaperRun_comparison_3runs/`.


In [ ]:
import os, glob as _glob, json as _json
from datetime import datetime

# ================= RUN SELECTION =================
# 100 = SmartHealth (4ch) | 300 = SmartHealth+DirVsAvg (4ch) | 310 = combined MD+DWI (2ch, BASELINE)
DATASETS = {
    100: dict(crop=110, rvip=105, channels=(0, 1, 2, 3), label='SmartHealth | ResEnc-M'),
    300: dict(crop=111, rvip=107, channels=(0, 1, 2, 3), label='SmartHealth+DirVsAvg | ResEnc-M'),
    310: dict(crop=111, rvip=107, channels=(0, 1),       label='Combined DWI+MD (baseline) | ResEnc-M'),
}
SHORT_LABELS = {                   # x-axis names in the plots
    100: 'SmartHealth\n4 contrasts',
    300: 'SmartHealth+DirVsAvg\n4 contrasts',
    310: 'SmartHealth+DirVsAvg\nDWI+MD (shipped)',
}
DATASET = 310                      # <-- the ONLY line to change per run (100 / 300 / 310)

# ================= STAGE TOGGLES (stop anywhere) =================
RUN_BUILD_GT = True                # 1. build GT masks (needs annotated GT) -- off for inference-only
RUN_CROP     = True                # 2. YOLO crop -> resize 256 -> normalize
RUN_LV       = True                # 3. nnUNet LV 5-fold ensemble
RUN_IP       = True                # 4. YOLO insertion points -> combine with LV
RUN_METRICS  = True                # 6. Dice / Hausdorff / MD / FA / DWI  (needs GT)
RUN_HELIX    = True                # 7. helix angle via CarDpy (needs raw diffusion + bvals/bvecs)
RUN_PLOTS    = True                # 8. comparison plots + GT-vs-pred overlays (needs GT)

# ================= EVALUATION FIXES =================
EVAL_LV_ON_PURE_MASK = True        # BUGFIX #1 (overwrite): LV metrics on pure LV; IPs never erase LV
VOXEL_SPACING_MM     = 1.25        # BUGFIX #2 (spacing): mm/voxel for Hausdorff (was hardcoded in eval_util)

# ================= COLLABORATOR MODE =================
COLLAB_MODE = False                # True -> use the paths/weights below instead of our Ultron layout
COLLAB = dict(
    test_data_root     = '/path/to/your/TestData',     # Volunteer_*/Distortion_Corrected/DiVO_*/{02,03,06}
    images_ts          = '/path/to/your/imagesTs',     # *_0000.nii.gz define the test cases
    crop_weights       = '/path/to/crop/weights/best.pt',
    rvip_weights       = '/path/to/rvip/weights/best.pt',
    lv_results_dir     = '/path/to/nnUNet_results',     # holds Dataset310_*/<trainer>__<plans>__2d
    lv_dataset_id      = 310, lv_channels = (0, 1),     # shipped baseline = Dataset310 (MD+DWI)
    raw_diffusion_root = '/path/to/raw/by/volunteer',   # for helix: <root>/Volunteer_*/Distortion_Corrected/DiVO_*/NifTis
    annotator          = 'Hannum',
)

# ================= ENV / PATHS =================
NNUNET_ROOT = os.environ.get('NNUNET_ROOT', os.getcwd())   # repo root (holds nnUNet_raw/_results, yolo_datasets)
os.environ['nnUNet_raw']          = f'{NNUNET_ROOT}/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = f'{NNUNET_ROOT}/nnUNet_preprocessed'
os.environ['nnUNet_results']      = f'{NNUNET_ROOT}/nnUNet_results'
os.environ['nnUNet_compile']      = 'false'                 # torch.compile was the training-collapse bug
YOLO_RUNS   = f'{NNUNET_ROOT}/yolo_datasets/runs'
CARDPY_PATH = NNUNET_ROOT                                   # import cardpy via sys.path.append(CARDPY_PATH)

trainer    = 'nnUNetTrainerDA5_100epochsNoCompile'         # all LV models retrained compile-off
plans      = 'nnUNetResEncUNetMPlans'
lv_config  = '2d'
folds      = '0 1 2 3 4'                                    # full 5-fold ensemble
checkpoint = 'checkpoint_final.pth'
RUN_TAG    = 'PaperRun'
ALLOW_OVERWRITE = False                                     # True -> clear an existing base_out and rerun

# ---- resolve the active run config ----
_cfg = DATASETS[DATASET]
if COLLAB_MODE:
    dataset_id_lv = COLLAB['lv_dataset_id']; LV_CHANNELS = tuple(COLLAB['lv_channels'])
    crop_yolo_id = rvip_yolo_id = 0
    run_label = f'Collaborator | LV D{dataset_id_lv}'
else:
    dataset_id_lv = DATASET; LV_CHANNELS = _cfg['channels']
    crop_yolo_id, rvip_yolo_id = _cfg['crop'], _cfg['rvip']; run_label = _cfg['label']

def _resolve_best(cands, what):
    for p in cands:
        if os.path.exists(p):
            print(f'{what}: {p}'); return p
    print(f'!! {what}: none of {cands} exist (ok if that stage is OFF).')
    return cands[0]

if COLLAB_MODE:
    YOLO_CROP_WEIGHTS     = COLLAB['crop_weights']
    YOLO_RVIP_WEIGHTS_AVG = COLLAB['rvip_weights']
else:
    YOLO_CROP_WEIGHTS = _resolve_best([f'{YOLO_RUNS}/crop_D{crop_yolo_id}/weights/best.pt'], 'crop model')
    YOLO_RVIP_WEIGHTS_AVG = _resolve_best([
        f'{YOLO_RUNS}/rvip_1c_D{rvip_yolo_id}/weights/best.pt',     # single-contrast
        f'{YOLO_RUNS}/rvip_D{rvip_yolo_id}_0-0-0/weights/best.pt',  # ablation (0-0-0)
        f'{YOLO_RUNS}/rvip_D{rvip_yolo_id}/weights/best.pt',        # baseline (0,0,0)
    ], 'single-contrast RVIP model')

RVIP_SINGLE_CLASS = True
RVIP_FLIP_ASSIGN  = False
RVIP_CONF         = 0.01
IP_METHODS = [{'tag': 'yolo_avg', 'kind': 'yolo', 'weights': YOLO_RVIP_WEIGHTS_AVG, 'channels': (0, 0, 0)}]

# ================= DATA =================
if COLLAB_MODE:
    IMAGES_TS          = COLLAB['images_ts']
    TEST_DATA_ROOT     = COLLAB['test_data_root']
    annotator          = COLLAB['annotator']
    RAW_DIFFUSION_ROOT = COLLAB['raw_diffusion_root']
    test_dataset_id    = dataset_id_lv
else:
    test_dataset_id = 100                                           # fixed SmartHealth holdout (SAME for every run)
    _ts_raw = _glob.glob(f"{os.environ['nnUNet_raw']}/Dataset{test_dataset_id:03d}_*")
    assert _ts_raw, f'no nnUNet_raw/Dataset{test_dataset_id:03d}_* folder found'
    IMAGES_TS          = f'{_ts_raw[0]}/imagesTs'
    TEST_DATA_ROOT     = f'{NNUNET_ROOT}/HannumSmartHealthTestData'
    annotator          = os.path.basename(TEST_DATA_ROOT)
    RAW_DIFFUSION_ROOT = f'{NNUNET_ROOT}/TestRawCases'              # raw NifTis (Interpolated.nii/.bvals/.bvecs/.header)
main_folder = os.path.dirname(TEST_DATA_ROOT)
HELIX_CACHE = f'{NNUNET_ROOT}/helix_eigvec_cache'    # DTI fit per raw slice, shared by GT/pred and all runs

# ================= OUTPUTS =================
run       = f'PaperRun_crop{crop_yolo_id}_rvip{rvip_yolo_id}_lv{dataset_id_lv}_{RUN_TAG}'
base_out  = f'{NNUNET_ROOT}/inferenceTest_{run}'
output_crop_masks   = f'{base_out}/CropInference'
output_image_folder = f'{base_out}/processed256/imagesTr'
output_LV           = f'{base_out}/OutputLV'
gt_folder           = f'{base_out}/GT_Combined'
gt_folder_lv        = f'{base_out}/GT_Combined_LV'
inspection_folder   = f'{base_out}/inspection'

# ---- stage-aware preflight (only check what a selected stage needs) ----
if RUN_LV:
    _md = (COLLAB['lv_results_dir'] if COLLAB_MODE else os.environ['nnUNet_results'])
    _model_dirs = _glob.glob(f'{_md}/Dataset{dataset_id_lv:03d}_*/{trainer}__{plans}__{lv_config}')
    assert len(_model_dirs) == 1, f'Expected one LV model folder, got {_model_dirs}'
    model_dir = _model_dirs[0]
    for fold in folds.split():
        assert os.path.isfile(f'{model_dir}/fold_{fold}/{checkpoint}'), f'Missing fold {fold} checkpoint'
    with open(f'{model_dir}/dataset.json') as _f:
        assert len(_json.load(_f)['channel_names']) == len(LV_CHANNELS), 'Model/input channel mismatch'
    print('LV model:', model_dir, '| channels:', LV_CHANNELS)
if RUN_CROP:
    assert os.path.isfile(YOLO_CROP_WEIGHTS), f'Missing crop weights: {YOLO_CROP_WEIGHTS}'
if RUN_IP:
    assert os.path.isfile(YOLO_RVIP_WEIGHTS_AVG), f'Missing RVIP weights: {YOLO_RVIP_WEIGHTS_AVG}'

if os.path.exists(base_out) and not ALLOW_OVERWRITE:
    raise AssertionError(f'Output already exists: {base_out}. Bump RUN_TAG or set ALLOW_OVERWRITE=True.')
for d in [output_crop_masks, output_image_folder, output_LV, gt_folder, gt_folder_lv, inspection_folder]:
    os.makedirs(d, exist_ok=True)

def clear_dirs(dirs):
    for d in dirs:
        if os.path.isdir(d):
            for f in os.listdir(d):
                p = os.path.join(d, f)
                if os.path.isfile(p) or os.path.islink(p): os.unlink(p)

print(f'RUN         : {run_label}')
print(f'crop/rvip/lv: D{crop_yolo_id} / D{rvip_yolo_id} / D{dataset_id_lv}   test cases: D{test_dataset_id}')
print(f'stages      : GT={RUN_BUILD_GT} CROP={RUN_CROP} LV={RUN_LV} IP={RUN_IP} METRICS={RUN_METRICS} HELIX={RUN_HELIX} PLOTS={RUN_PLOTS}')
print(f'outputs     : {base_out}')


In [ ]:
import glob, re
import numpy as np
import nibabel as nib
import cv2
import pandas as pd
import matplotlib.pyplot as plt

import yolo_pipeline as yp
from eval_util import process_folders, dice_score_original, upscale_segmentation_to_original

interpolation_size = 256

def normalize_avg_data(x):         return (x - np.min(x)) / (np.max(x) - np.min(x))
def normalize_mean_diff_data(x):   return (x - 0) / 4.0
def normalize_eigenvector_data(x): return x / np.sqrt(2)   # X+Y capped at sqrt(2) (unit eigenvector)

def make_square_crop(mask, target_size=256):
    coords = np.argwhere(mask > 0)
    x_min, y_min = coords.min(axis=0); x_max, y_max = coords.max(axis=0) + 1
    side = max(x_max - x_min, y_max - y_min)
    cx, cy = (x_min + x_max) // 2, (y_min + y_max) // 2
    x_min = max(0, cx - side // 2); x_max = x_min + side
    y_min = max(0, cy - side // 2); y_max = y_min + side
    return x_min, x_max, y_min, y_max, side / target_size

def parse_case(fname):
    stem = re.sub(r'(_0000)?\.nii(\.gz)?$', '', fname)
    parts = stem.split('_'); i = parts.index('Volunteer'); j = parts.index('slice')
    return f'{parts[i]}_{parts[i+1]}', f'{parts[i+2]}_{parts[i+3]}_{parts[i+4]}', f'{int(parts[j+1]):03d}'

def case_id(volunteer, divo, slc):  return f'{annotator}_{volunteer}_{divo}_slice_{slc}'
def source_dir(volunteer, divo):    return os.path.join(TEST_DATA_ROOT, volunteer, 'Distortion_Corrected', divo)

cases = [parse_case(os.path.basename(f)) for f in sorted(glob.glob(os.path.join(IMAGES_TS, '*_0000.nii.gz')))]
print(f'{len(cases)} test cases from {IMAGES_TS}')
[print('  ', c) for c in cases[:8]]


## 1. Build ground-truth masks (combined LV=1/IP1=2/IP2=3 + pure LV) — needs annotated GT

In [ ]:
if RUN_BUILD_GT:
    def combine_gt_mask(mask3):
        out = np.zeros(mask3.shape[:2], dtype=np.uint8)
        out[mask3[:, :, 0] > 0] = 1; out[mask3[:, :, 1] > 0] = 2; out[mask3[:, :, 2] > 0] = 3
        return out
    def combine_gt_mask_lv(mask3):        # pure LV (label 1) -- NOT eroded by IPs -> correct LV metrics
        out = np.zeros(mask3.shape[:2], dtype=np.uint8); out[mask3[:, :, 0] > 0] = 1; return out

    clear_dirs([gt_folder, gt_folder_lv]); built = 0
    for volunteer, divo, slc in cases:
        gt_src = os.path.join(source_dir(volunteer, divo), '06_Segmentation_Masks_CI',
                              f'Cropped_Segmentation_Slice_{slc}.nii')
        if not os.path.exists(gt_src): print('missing GT mask:', gt_src); continue
        m = nib.load(gt_src); _md = m.get_fdata(); cid = case_id(volunteer, divo, slc)
        nib.save(nib.Nifti1Image(combine_gt_mask(_md),    m.affine), os.path.join(gt_folder,    cid + '.nii.gz'))
        nib.save(nib.Nifti1Image(combine_gt_mask_lv(_md), m.affine), os.path.join(gt_folder_lv, cid + '.nii.gz'))
        built += 1
    print(f'built {built} GT masks (combined + pure-LV) -> {gt_folder} , {gt_folder_lv}')
else:
    print('RUN_BUILD_GT=False -> skipping GT build (inference-only).')


## 2. YOLO crop → square crop → resize 256 → per-contrast normalize

In [ ]:
if RUN_CROP:
    crop_model = yp.load_model(YOLO_CROP_WEIGHTS)
    clear_dirs([output_crop_masks, output_image_folder]); done = 0
    for volunteer, divo, slc in cases:
        img_folder = os.path.join(source_dir(volunteer, divo), '03_Segmentation_Images')
        avg_f = os.path.join(img_folder, f'Average_Diffusion_Weighted_Image_Slice_{slc}.nii')
        md_f  = os.path.join(img_folder, f'Mean_Diffusivty_Image_Slice_{slc}.nii')
        eig_f = os.path.join(img_folder, f'Primary_Eigenvector_Image_Slice_{slc}.nii')
        fa_f  = os.path.join(img_folder, f'Fractional_Anisotropy_Image_Slice_{slc}.nii')
        if not all(os.path.exists(p) for p in [avg_f, md_f, eig_f, fa_f]):
            print('missing modality for', volunteer, divo, slc); continue
        avg_img = nib.load(avg_f); avg = avg_img.get_fdata()
        md_ = nib.load(md_f).get_fdata(); eig = nib.load(eig_f).get_fdata(); fa = nib.load(fa_f).get_fdata()

        box = yp.detect_crop_box(crop_model, avg)
        if box is None: print('no crop detected -> skipping', volunteer, divo, slc); continue
        crop_mask = yp.crop_box_to_mask(box, avg.shape); cid = case_id(volunteer, divo, slc)
        nib.save(nib.Nifti1Image(crop_mask.astype(np.uint8), avg_img.affine),
                 os.path.join(output_crop_masks, cid + '.nii.gz'))

        x0, x1, y0, y1, _ = make_square_crop(crop_mask)
        avg_c = normalize_avg_data(       cv2.resize(avg[x0:x1, y0:y1],                              (256,256), interpolation=cv2.INTER_LINEAR))
        md_c  = normalize_mean_diff_data( cv2.resize(md_[x0:x1, y0:y1],                              (256,256), interpolation=cv2.INTER_LINEAR))
        eig_c = normalize_eigenvector_data(cv2.resize(eig[x0:x1, y0:y1, 0] + eig[x0:x1, y0:y1, 1],  (256,256), interpolation=cv2.INTER_LINEAR))
        fa_c  =                           cv2.resize(fa[x0:x1, y0:y1],                               (256,256), interpolation=cv2.INTER_LINEAR)
        aff = np.eye(4); _chan = [avg_c, md_c, eig_c, fa_c]
        for ch in LV_CHANNELS:                 # 4 for 100/300; (0,1)=avg+MD for the 310 DWI+MD model
            nib.save(nib.Nifti1Image(_chan[ch], aff), os.path.join(output_image_folder, f'{cid}_{ch:04d}.nii.gz'))
        done += 1
    print(f'crop + resize done for {done} cases -> {output_image_folder}')
else:
    print('RUN_CROP=False -> skipping (expects pre-made 256 inputs in', output_image_folder, ').')


## 3. nnUNet LV segmentation — 5-fold ensemble (compile off)

In [ ]:
if RUN_LV:
    import subprocess, shlex, sys
    from pathlib import Path
    _predict = Path(sys.executable).parent / 'nnUNetv2_predict'
    assert _predict.is_file(), f'Select the nnunet Conda kernel; missing {_predict}'
    clear_dirs([output_LV])
    cmd = [str(_predict), '-i', output_image_folder, '-o', output_LV,
           '-d', str(dataset_id_lv), '-c', lv_config, '-tr', trainer, '-p', plans,
           '-f', *folds.split(), '-chk', checkpoint]
    print(shlex.join(cmd))
    subprocess.run(cmd, check=True, env=dict(os.environ, nnUNet_compile='false'))
else:
    print('RUN_LV=False -> skipping LV inference.')


## 4–6. Insertion points → combine → metrics
`combine()` writes LV=1 then IP=2/3 **into a copy** (`Output_Final_Combined_*`); the standalone LV file
(`OutputLV`) stays pure. With `EVAL_LV_ON_PURE_MASK=True`, LV Dice/MD/FA/DWI are scored on `OutputLV`
vs the pure-LV GT (IPs can't erase LV), and IP Hausdorff (labels 2/3) comes from the combined mask.
`VOXEL_SPACING_MM` sets the mm/voxel conversion (dynamic-spacing fix).

In [ ]:
def detect_ips_yolo(m, ip_dir):
    model = yp.load_model(m['weights'])
    for volunteer, divo, slc in cases:
        cid = case_id(volunteer, divo, slc)
        img0 = os.path.join(output_image_folder, f'{cid}_0000.nii.gz')
        if not os.path.exists(img0): continue
        pts = yp.detect_rvips_from_path(model, img0, channels=m['channels'], single_class=RVIP_SINGLE_CLASS,
                                        flip_assignment=RVIP_FLIP_ASSIGN, conf_thresh=RVIP_CONF)
        ipm = yp.rvips_to_mask(pts, (256, 256), anterior_label=2, inferior_label=3)
        nib.save(nib.Nifti1Image(ipm.astype(np.uint8), np.eye(4)), os.path.join(ip_dir, cid + '.nii.gz'))

def detect_ips_nnunet(m, ip_dir):
    cmd = (f"nnUNetv2_predict -i {output_image_folder} -o {ip_dir} -d {m['dataset']} "
           f"-c {m['config']} -tr {m['trainer']} -p {m['plans']} -f {m['folds']}")
    print(cmd); os.system(cmd)

def combine(kind, ip_dir, final_dir):
    for f in os.listdir(output_LV):
        if not f.endswith('.nii.gz'): continue
        lv_img = nib.load(os.path.join(output_LV, f)); lv = lv_img.get_fdata().astype(np.int32)
        out = lv.copy(); out[lv == 1] = 1
        ip_path = os.path.join(ip_dir, f)
        if os.path.exists(ip_path):
            ip = nib.load(ip_path).get_fdata().astype(np.int32)
            a, b = (2, 3) if kind == 'yolo' else (1, 2)   # yolo already 2/3; nnUNet 1/2 -> 2/3
            out[ip == a] = 2; out[ip == b] = 3            # overlapping LV becomes IP in this COPY only
        nib.save(nib.Nifti1Image(out, lv_img.affine, lv_img.header), os.path.join(final_dir, f))

def run_ip_method(m):
    tag = m['tag']
    ip_dir    = f'{base_out}/OutputIPs_{tag}'
    final_dir = f'{base_out}/Output_Final_Combined_{tag}'
    for d in (ip_dir, final_dir): os.makedirs(d, exist_ok=True)
    clear_dirs([ip_dir, final_dir])
    (detect_ips_yolo if m['kind'] == 'yolo' else detect_ips_nnunet)(m, ip_dir)
    combine(m['kind'], ip_dir, final_dir)
    df = None
    if RUN_METRICS:
        if EVAL_LV_ON_PURE_MASK:
            lv_rows, _t1 = [], []
            process_folders(output_LV, gt_folder_lv, main_folder, lv_rows, _t1, None, annotator,
                            output_crop_masks, voxel_spacing_mm=VOXEL_SPACING_MM)
            ip_rows, _t2 = [], []
            process_folders(final_dir, gt_folder, main_folder, ip_rows, _t2, None, annotator,
                            output_crop_masks, voxel_spacing_mm=VOXEL_SPACING_MM)
            df = pd.DataFrame(lv_rows); ip_df = pd.DataFrame(ip_rows).set_index('Case ID')
            for col in ('Hausdorff Distance Label 2', 'Hausdorff Distance Label 3'):
                df[col] = df['Case ID'].map(ip_df[col])
            eroded = []
            for f in sorted(os.listdir(output_LV)):
                if not f.endswith('.nii.gz'): continue
                lvm = nib.load(os.path.join(output_LV, f)).get_fdata()
                cmb = nib.load(os.path.join(final_dir, f)).get_fdata()
                eroded.append(int(((lvm == 1) & (cmb != 1)).sum()))
            print(f'[{tag}] overwrite fix ON: LV pixels an IP would erase = mean {np.mean(eroded):.1f}, '
                  f'max {int(np.max(eroded)) if eroded else 0} -> excluded from LV metrics.')
        else:
            md_data, mm = [], []
            print('[legacy] LV metrics on the IP-overwritten combined mask (overwrite NOT fixed).')
            process_folders(final_dir, gt_folder, main_folder, md_data, mm, None, annotator,
                            output_crop_masks, voxel_spacing_mm=VOXEL_SPACING_MM)
            df = pd.DataFrame(md_data)
        xlsx = f'{base_out}/{run}_{tag}.xlsx'; df.to_excel(xlsx, index=False)
        df.to_csv(xlsx.replace('.xlsx', '.csv'), index=False)
        print(f'[{tag}] {len(df)} cases -> {xlsx}')
    return tag, df

results = {}
if RUN_IP:
    for m in IP_METHODS:
        print('=' * 30, m['tag'], '=' * 30)
        tag, df = run_ip_method(m)
        if df is not None: results[tag] = df
else:
    print('RUN_IP=False -> skipping insertion points + metrics.')


## 7. IP-method comparison summary

In [ ]:
if RUN_METRICS and results:
    cols = ['Dice Score Original Label 1', 'Hausdorff Distance Label 2', 'Hausdorff Distance Label 3',
            'GT_Median_MD', 'Pred_median_MD', 'GT_Median_FA', 'Pred_median_FA', 'Avg. HD Epi', 'Avg. HD Endo']
    summary = pd.DataFrame({tag: df[cols].mean() for tag, df in results.items()})
    print(summary)
    summary.to_excel(f'{base_out}/{run}_IPmethod_comparison.xlsx')
    summary.to_csv(f'{base_out}/{run}_IPmethod_comparison.csv')
    # Hausdorff ~ VOXEL_SPACING_MM*1000 = eval failsafe for a MISSED point; a good detector is single-digit px.
else:
    print('no metrics to summarize (RUN_METRICS off or no results).')


## 8. Un-crop predictions back to original field-of-view (for helix + overlays)

In [ ]:
def build_fullfov_lv(dst_dir):
    """Upscale the pure LV prediction (label 1) from the 256 crop back to the original image size,
    using the YOLO crop box -- needed by the helix stage (must align with the raw diffusion data)."""
    os.makedirs(dst_dir, exist_ok=True); clear_dirs([dst_dir]); n = 0
    for volunteer, divo, slc in cases:
        cid = case_id(volunteer, divo, slc)
        lv_p   = os.path.join(output_LV, cid + '.nii.gz')
        crop_p = os.path.join(output_crop_masks, cid + '.nii.gz')
        if not (os.path.exists(lv_p) and os.path.exists(crop_p)): continue
        lv256 = (nib.load(lv_p).get_fdata() == 1).astype(np.uint8)     # pure LV
        full = upscale_segmentation_to_original(lv256, crop_p)          # -> original FoV
        full = np.asarray(full).astype(np.uint8)
        if full.ndim == 2: full = full[:, :, None]                      # (H,W,1) so sitk[:, :, 0] works
        nib.save(nib.Nifti1Image(full, np.eye(4)), os.path.join(dst_dir, cid + '.nii.gz')); n += 1
    print(f'built {n} full-FoV predicted LV masks -> {dst_dir}')
    return dst_dir

pred_lv_fullfov = f'{base_out}/Pred_LV_fullFoV'
if RUN_HELIX or RUN_PLOTS:
    build_fullfov_lv(pred_lv_fullfov)


## 9. Helix angle (CarDpy) — HAP / HAR on predicted vs ground-truth LV
Needs the raw 4D diffusion per case: `RAW_DIFFUSION_ROOT/<Volunteer>/Distortion_Corrected/<DiVO>/NifTis`
with `.nii`, `.header`, `.bvals`, `.bvecs`. GT uses `06_Segmentation_Masks_CI`; predicted uses the
upscaled full-FoV LV. Compares HAP/HAR per case.

In [ ]:
if RUN_HELIX:
    import sys as _sys
    if CARDPY_PATH not in _sys.path: _sys.path.append(CARDPY_PATH)
    from sklearn.linear_model import LinearRegression
    from cardpy.Data_Import import NifTi_Reader
    from cardpy.Data_Sorting import stacked2sorted
    from cardpy.Data_Processing.DTI import DTI_recon
    from cardpy.Data_Processing.cDTI import cDTI_recon, Endo2Epi_Grid

    NUM_INTERP_POINTS = 200; SMOOTHNESS = 'Native'
    HA_FILTER = {'Linear Filter: Outlier StDev': 1,
                 'Spatial Filter: Wall Depth Factor': 0.25,
                 'Spatial Filter: Kernel Size': 5}

    os.makedirs(HELIX_CACHE, exist_ok=True)

    def eigenvectors_for(nifti_folder, volunteer, slice_idx):
        # DTI fit of one raw slice (~10 s) -- independent of the mask, so cached on disk
        cache = os.path.join(HELIX_CACHE, f'{volunteer}_slice{slice_idx:03d}.npz')
        if os.path.exists(cache):
            z = np.load(cache); return {k: z[k] for k in z.files}
        nii = glob.glob(os.path.join(nifti_folder, '*.nii')); hdr = glob.glob(os.path.join(nifti_folder, '*.header'))
        bva = glob.glob(os.path.join(nifti_folder, '*.bvals')); bve = glob.glob(os.path.join(nifti_folder, '*.bvecs'))
        if not (nii and hdr and bva and bve): return None
        [mat, bvals, bvecs, Header, _, _] = NifTi_Reader(nii[0], bva[0], bve[0], hdr[0])
        if mat.ndim != 4: return None
        mat = mat[:, :, slice_idx, :][:, :, np.newaxis, :]
        [ms, bvs, bvc] = stacked2sorted(mat, bvals, bvecs)
        [_, _, Eigenvectors, _] = DTI_recon(ms, bvs, bvc, tensor_fit='NLLS')
        np.savez(cache, **Eigenvectors)
        return Eigenvectors

    def hap_for(nifti_folder, mask_path, volunteer, slice_idx):
        try:
            Eigenvectors = eigenvectors_for(nifti_folder, volunteer, slice_idx)
            if Eigenvectors is None or not os.path.exists(mask_path): return None
            # nibabel, NOT SimpleITK: sitk's array is transposed vs CarDpy's NifTi_Reader (checked against the MD map)
            myo = np.asarray(nib.load(mask_path).get_fdata())
            myo = (np.nan_to_num(myo if myo.ndim == 2 else myo[:, :, 0]) > 0).astype(np.uint8)[:, :, np.newaxis]
            [cDTI_Metrics, Epi, Endo, Mask] = cDTI_recon(myo, Eigenvectors, NUM_INTERP_POINTS, SMOOTHNESS, HA_FILTER)
            mask_nan = np.copy(Mask).astype('float'); mask_nan[mask_nan == 0] = np.nan
            grid = Endo2Epi_Grid(np.copy(Mask)) * mask_nan; grid = np.clip(grid, 0.0, 1)
            HAPs, HARs = [], []
            for s in range(mask_nan.shape[2]):
                nm = ~np.isnan(grid[:, :, s].flatten())
                e2e = grid[:, :, s].flatten()[nm]
                ha  = cDTI_Metrics['HASF'][:, :, s].flatten()[nm]  # spatially-filtered HA (HASF)
                if len(e2e) == 0 or len(ha) == 0: continue
                reg = LinearRegression().fit(e2e[:, None], ha[:, None])
                xp = np.linspace(0, 1, 101); yp = reg.intercept_ + reg.coef_ * xp[:, None]
                HAPs.append(float(reg.coef_[0])); HARs.append(float(yp[0] - yp[-1]))
            return (float(np.mean(HAPs)) if HAPs else np.nan, float(np.mean(HARs)) if HARs else np.nan)
        except Exception as e:
            print('   HA error:', e); return None

    rows = []
    for volunteer, divo, slc in cases:
        cid = case_id(volunteer, divo, slc); si = int(slc) - 1
        nifti_folder = os.path.join(RAW_DIFFUSION_ROOT, volunteer, 'Distortion_Corrected', divo, 'NifTis')
        # GT = native full-FoV manual mask (06_*_CI is in the 256-crop space, not aligned with the raw data)
        gt_mask   = os.path.join(source_dir(volunteer, divo), '04_Segmentation_Masks', f'Segmentation_Slice_{slc}.nii')
        pred_mask = os.path.join(pred_lv_fullfov, cid + '.nii.gz')
        gt  = hap_for(nifti_folder, gt_mask, volunteer, si)   if os.path.exists(gt_mask)   else None
        pr  = hap_for(nifti_folder, pred_mask, volunteer, si) if os.path.exists(pred_mask) else None
        rows.append({'Case ID': cid, 'volunteer': volunteer, 'divo': divo, 'slice': slc,
                     'HAP_gt':  gt[0] if gt else np.nan, 'HAR_gt':  gt[1] if gt else np.nan,
                     'HAP_pred': pr[0] if pr else np.nan, 'HAR_pred': pr[1] if pr else np.nan})
        print(f'  {cid}: HAP gt={rows[-1]["HAP_gt"]}, pred={rows[-1]["HAP_pred"]}')
    ha_df = pd.DataFrame(rows)
    ha_df['HAP_abs_err'] = (ha_df['HAP_pred'] - ha_df['HAP_gt']).abs()
    ha_df['HAR_abs_err'] = (ha_df['HAR_pred'] - ha_df['HAR_gt']).abs()
    ha_csv = f'{base_out}/{run}_helix.csv'; ha_df.to_csv(ha_csv, index=False)
    print(f'helix: {ha_df["HAP_pred"].notna().sum()} cases -> {ha_csv}')
    print(ha_df[['HAP_gt', 'HAP_pred', 'HAP_abs_err', 'HAR_gt', 'HAR_pred', 'HAR_abs_err']].agg(['mean', 'median']).round(2))
else:
    print('RUN_HELIX=False -> skipping helix angle (needs raw diffusion data + bvals/bvecs).')


## 10. Comparison plots + CSVs (Dice / IP Hausdorff / MD / FA / DWI / helix) — this run, then all 3 runs

In [ ]:
def run_table(ds):
    """Saved per-case metrics of one run (+ helix columns if that run's helix CSV exists), or None."""
    c = DATASETS[ds]; r = f'PaperRun_crop{c["crop"]}_rvip{c["rvip"]}_lv{ds}_{RUN_TAG}'
    xlsx = f'{NNUNET_ROOT}/inferenceTest_{r}/{r}_{IP_METHODS[-1]["tag"]}.xlsx'
    if not os.path.exists(xlsx): return None
    df = pd.read_excel(xlsx)
    ha_csv = f'{NNUNET_ROOT}/inferenceTest_{r}/{r}_helix.csv'
    if os.path.exists(ha_csv):
        ha = pd.read_csv(ha_csv).rename(columns={'HAP_gt': 'GT_HAP', 'HAP_pred': 'Pred_HAP',
                                                 'HAR_gt': 'GT_HAR', 'HAR_pred': 'Pred_HAR'})
        ha['Case ID'] = ha['Case ID'].str.replace(f'^{annotator}_', '', regex=True)
        df['_key'] = df['Case ID'].str.replace(f'^{annotator}_', '', regex=True)
        df = df.merge(ha[['Case ID', 'GT_HAP', 'Pred_HAP', 'GT_HAR', 'Pred_HAR']].rename(columns={'Case ID': '_key'}),
                      on='_key', how='left').drop(columns='_key')
    return df

if RUN_PLOTS and not COLLAB_MODE:
    import importlib, comparison_plots as cp
    importlib.reload(cp)
    _this = run_table(DATASET)
    if _this is None:
        print('plots skipped: no saved metrics for this run (needs RUN_METRICS once).')
    else:
        print(f'run_label   : {run_label}')
        print(f'crop/rvip/lv: D{crop_yolo_id} / D{rvip_yolo_id} / D{dataset_id_lv}   (test cases: D{test_dataset_id})')
        cp.full_report({SHORT_LABELS[DATASET]: _this}, f'{base_out}/comparison')
        _all = {SHORT_LABELS[d]: run_table(d) for d in DATASETS}
        if all(v is not None for v in _all.values()):
            cmp_dir = f'{NNUNET_ROOT}/PaperRun_comparison_3runs'
            print(f'\n===== all 3 runs -> {cmp_dir} =====')
            alldf, summary = cp.full_report(_all, cmp_dir)
        else:
            print('3-run comparison waits for:', [k for k, v in _all.items() if v is None])
elif RUN_PLOTS and RUN_METRICS and results:
    import importlib, comparison_plots as cp
    importlib.reload(cp)
    runs = {f'{run_label} | {m["tag"]}': f'{base_out}/{run}_{m["tag"]}.xlsx' for m in IP_METHODS}
    alldf, summary = cp.full_report(runs, f'{base_out}/comparison')
else:
    print('plots skipped (need RUN_PLOTS + saved metrics).')


## 11. GT-vs-prediction overlays (full view + cropped, LV + IPs)

In [ ]:
if RUN_PLOTS and RUN_BUILD_GT and results:
    import importlib, comparison_plots as cp
    importlib.reload(cp)
    _ip_tag   = IP_METHODS[-1]['tag']
    _pred_dir = f'{base_out}/Output_Final_Combined_{_ip_tag}'
    _ovl_dir  = f'{base_out}/overlays_{_ip_tag}'; os.makedirs(_ovl_dir, exist_ok=True)
    for volunteer, divo, slc in cases:
        cid = case_id(volunteer, divo, slc)
        pred_p = os.path.join(_pred_dir, cid + '.nii.gz'); gt_p = os.path.join(gt_folder, cid + '.nii.gz')
        crop_img_p = os.path.join(output_image_folder, f'{cid}_0000.nii.gz')
        yolo_crop_p = os.path.join(output_crop_masks, cid + '.nii.gz')
        if not all(os.path.exists(x) for x in (pred_p, gt_p, crop_img_p, yolo_crop_p)): continue
        pred256 = nib.load(pred_p).get_fdata(); gt256 = nib.load(gt_p).get_fdata()
        crop_img = nib.load(crop_img_p).get_fdata().squeeze()
        full_img = nib.load(os.path.join(source_dir(volunteer, divo), '03_Segmentation_Images',
                            f'Average_Diffusion_Weighted_Image_Slice_{slc}.nii')).get_fdata().squeeze()
        gt_crop_p = os.path.join(source_dir(volunteer, divo), '02_Crop_Masks', f'Square_Crop_Mask_Slice_{slc}.nii')
        full_pred = upscale_segmentation_to_original(pred256, yolo_crop_p)
        full_gt   = upscale_segmentation_to_original(gt256,  gt_crop_p)
        cp.save_case_overlay(os.path.join(_ovl_dir, cid + '.png'),
                             full_img, full_gt, full_pred, crop_img, gt256, pred256, title=cid)
    print(f'saved GT-vs-pred overlays -> {_ovl_dir}')
else:
    print('overlays skipped (need RUN_PLOTS + RUN_BUILD_GT + results).')
